Esta clase se centrará en enriquecer el entorno visual y el *feedback* del juego mediante la adición de fondos con `parallax scrolling` (desplazamiento diferencial), la capacidad de usar secuencias de video como fondos animados y la implementación de un sistema de partículas para efectos como explosiones.

-----

### **Clase: Fondos Dinámicos - Parallax, Video y Partículas**

**Objetivo:** Extender el motor para gestionar fondos complejos. Esto incluye la creación de un sistema de `Parallax Scrolling` multicapa, un método para cargar videos como fondos animados y la implementación de un `ParticleEmitter` (Emisor de Partículas) para efectos visuales.

-----

### **1. Discusión Teórica: Simulación de Profundidad**

En un juego 2D, la profundidad es una ilusión. El **Parallax Scrolling** es la técnica principal para crear esta ilusión.

  * **Concepto:** Se dibujan múltiples capas de fondo (cielo, montañas, árboles, suelo) y se mueven a diferentes velocidades en relación con la cámara.
  * **Regla:** Las capas más lejanas (como el cielo o las nubes) se mueven más lento. Las capas más cercanas (como los arbustos en primer plano) se mueven más rápido.
  * **Implementación:** Logramos esto asignando un `scroll_factor` (factor de desplazamiento) a cada capa. Un factor de `0.0` significa que la capa no se mueve (el cielo). Un factor de `1.0` significa que se mueve a la misma velocidad que la cámara (el suelo del juego). Un factor de `0.5` la movería a la mitad de la velocidad de la cámara (montañas lejanas).

-----

### **2. Expansión del Motor: Nuevos Módulos**

Añadiremos dos nuevos módulos a la carpeta `engine/` y actualizaremos el `AssetManager`.

#### **`engine/asset_manager.py` (Actualizado)**

Añadimos un método para cargar una secuencia de imágenes (frames) desde una carpeta.

```python
# (Extracto del método a añadir en engine/asset_manager.py)
# ... dentro de la clase AssetManager ...

    def load_image_sequence(self, name, folder_path):
        """
        Carga una secuencia de imágenes (frames) desde una carpeta,
        ordenadas alfabéticamente. Útil para videos pre-renderizados.
        """
        frames = []
        try:
            # Ordenamos los archivos para asegurar el orden correcto de la animación
            filenames = sorted(os.listdir(folder_path))
            for filename in filenames:
                if filename.endswith(('.png', '.jpg', '.webp')):
                    image_path = os.path.join(folder_path, filename)
                    frames.append(pygame.image.load(image_path).convert_alpha())
            self.images[name] = frames # Guardamos la lista de frames
            print(f"Secuencia '{name}' cargada ({len(frames)} frames) desde '{folder_path}'")
        except Exception as e:
            print(f"Error al cargar la secuencia '{name}': {e}")
```

#### **`engine/background.py` (¡NUEVO\!)**

Este módulo gestionará todos los tipos de fondos.

```python
# engine/background.py
import pygame

class ParallaxLayer:
    """Representa una única capa de fondo con su factor de scroll."""
    def __init__(self, image_surface, scroll_factor_x, scroll_factor_y):
        self.image = image_surface
        self.scroll_factor_x = scroll_factor_x
        self.scroll_factor_y = scroll_factor_y
        self.rect = self.image.get_rect()

    def update(self, camera_dx, camera_dy):
        """Mueve la capa basado en el movimiento de la cámara y su factor."""
        self.rect.x -= camera_dx * self.scroll_factor_x
        self.rect.y -= camera_dy * self.scroll_factor_y

    def draw(self, surface, camera_offset):
        # Dibuja la imagen en su posición, ajustada por el offset de la cámara
        # (Esto es simple, se puede mejorar con 'tiling' para fondos infinitos)
        surface.blit(self.image, (self.rect.x + camera_offset.x, self.rect.y + camera_offset.y))

class AnimatedBackgroundLayer(ParallaxLayer):
    """Una capa de fondo que es una animación (ej. un video pre-renderizado)."""
    def __init__(self, frames, scroll_factor_x, scroll_factor_y, anim_speed=0.5):
        self.frames = frames
        self.is_animated = True
        self.current_frame = 0
        self.anim_speed = anim_speed
        self.anim_timer = 0
        super().__init__(self.frames[0], scroll_factor_x, scroll_factor_y)

    def update(self, camera_dx, camera_dy):
        super().update(camera_dx, camera_dy)
        
        # Actualizar animación
        self.anim_timer += self.anim_speed
        if self.anim_timer >= 1:
            self.anim_timer = 0
            self.current_frame = (self.current_frame + 1) % len(self.frames)
            self.image = self.frames[self.current_frame]

class BackgroundManager:
    """Gestiona múltiples capas de fondo (estáticas, parallax, video)."""
    def __init__(self):
        self.layers = []

    def add_layer(self, layer):
        self.layers.append(layer)
        # Ordenamos las capas por su factor de scroll para dibujarlas del más lento al más rápido
        self.layers.sort(key=lambda l: (l.scroll_factor_x + l.scroll_factor_y) / 2)

    def update(self, camera_dx, camera_dy):
        """Actualiza la posición de todas las capas."""
        for layer in self.layers:
            layer.update(camera_dx, camera_dy)

    def draw(self, surface, camera_offset):
        """Dibuja todas las capas en orden, del fondo al frente."""
        for layer in self.layers:
            layer.draw(surface, camera_offset)
```

#### **`engine/particle.py` (¡NUEVO\!)**

Este módulo define las partículas y el emisor que las crea.

```python
# engine/particle.py
import pygame
import random

class Particle(pygame.sprite.Sprite):
    def __init__(self, x, y, color, size, lifespan, velocity):
        super().__init__()
        self.image = pygame.Surface((size, size))
        self.image.fill(color)
        self.rect = self.image.get_rect(center=(x, y))
        self.lifespan = lifespan
        self.creation_time = pygame.time.get_ticks()
        self.pos = pygame.math.Vector2(x, y)
        self.vel = pygame.math.Vector2(velocity)
        self.gravity = 0.2

    def update(self):
        self.vel.y += self.gravity
        self.pos += self.vel
        self.rect.center = self.pos
        
        if pygame.time.get_ticks() - self.creation_time > self.lifespan:
            self.kill() # Se elimina de todos los grupos

class ParticleEmitter:
    def __init__(self, sprite_group, camera_offset_func):
        self.particle_group = sprite_group
        # Una función que nos dice el offset de la cámara para dibujar
        self.get_camera_offset = camera_offset_func

    def explode(self, x, y, count=20):
        """Crea una explosión de partículas en una posición del mundo."""
        colors = [(255, 0, 0), (255, 128, 0), (255, 255, 0), (200, 200, 200)]
        for _ in range(count):
            vel_x = random.uniform(-5, 5)
            vel_y = random.uniform(-10, -3)
            size = random.randint(3, 7)
            lifespan = random.randint(500, 1500)
            color = random.choice(colors)
            particle = Particle(x, y, color, size, lifespan, (vel_x, vel_y))
            self.particle_group.add(particle)

    def update(self):
        self.particle_group.update()
        
    def draw(self, surface):
        # Dibujamos las partículas ajustadas por la cámara
        offset = self.get_camera_offset()
        for particle in self.particle_group:
            surface.blit(particle.image, particle.rect.move(offset))
```

-----

### **3. Integración en el Juego (`platformer/`)**

Modificamos `game.py` y `game_scene.py` para usar estos nuevos sistemas.

#### **Archivo `platformer/game.py` (Actualizado)**

*Cargamos los nuevos assets.*

```python
# platformer/game.py
# ... (imports existentes)
from engine.background import BackgroundManager # Importamos el nuevo gestor

class PlatformerGame:
    def __init__(self):
        # ... (código __init__ existente) ...
        self.assets = AssetManager()
        self.audio_manager = AudioManager()
        
        # --- NUEVO: Creamos el BackgroundManager ---
        self.background_manager = BackgroundManager()
        
        self.load_assets()
        # ... (código de escenas) ...

    def load_assets(self):
        try:
            sprites_dir = os.path.join(self.assets_dir, 'sprites')
            audio_dir = os.path.join(self.assets_dir, 'audio')
            bg_dir = os.path.join(self.assets_dir, 'backgrounds')
            video_frames_dir = os.path.join(self.assets_dir, 'video_frames')

            # Carga de sprites
            self.assets.load_spritesheet('player_ss', os.path.join(sprites_dir, 'mario.json'))
            
            # Carga de fondos (parallax)
            self.assets.load_image('bg_cielo', os.path.join(bg_dir, 'cielo.png'))
            self.assets.load_image('bg_montanas', os.path.join(bg_dir, 'montanas.png'))
            
            # Carga de video (como secuencia de frames)
            self.assets.load_image_sequence('video_fondo', video_frames_dir)
            
            # Carga de sonidos
            # ... (código de audio_manager.load_sound) ...

        except Exception as e:
            print(f"Error fatal al cargar assets: {e}"); self.running = False
    
    # ... (el resto de la clase run y change_scene no cambian) ...
```

#### **Archivo `platformer/scenes/game_scene.py` (Actualizado)**

*Implementamos los nuevos gestores.*

```python
# platformer/scenes/game_scene.py
import pygame
import os
from engine.ui import Scene
from engine.camera import Camera
from engine.tilemap import TilemapLoader
from engine.particle import ParticleEmitter
from engine.background import ParallaxLayer, AnimatedBackgroundLayer
from platformer.entities.player import Player

class GameScene(Scene):
    def __init__(self, game):
        super().__init__(game)
        
        # --- Carga de Nivel ---
        loader = TilemapLoader(self.game.assets)
        level_path = os.path.join(self.game.assets_dir, 'nivel_1.json')
        result = loader.load_level(level_path)
        if result is None: self.game.running = False; return
        self.sprite_groups, world_w, world_h, player_start_pos = result

        # --- Creación de Entidades ---
        self.jugador = Player(player_start_pos[0], player_start_pos[1], self.game.assets)
        self.sprite_groups["all_sprites"].add(self.jugador)
        
        # --- Configuración de la Cámara ---
        self.camara = Camera(self.game.screen.get_width(), self.game.screen.get_height(), world_w, world_h)
        self.last_camera_x = self.camara.camera_rect.x
        
        # --- Configuración del Fondo (Parallax) ---
        self.game.background_manager.add_layer(
            ParallaxLayer(self.game.assets.get_image('bg_cielo'), 0.1, 0.1)
        )
        self.game.background_manager.add_layer(
            ParallaxLayer(self.game.assets.get_image('bg_montanas'), 0.5, 0.5)
        )
        # Ejemplo de capa de video (se mueve con el suelo)
        # self.game.background_manager.add_layer(
        #     AnimatedBackgroundLayer(self.game.assets.get_image('video_fondo'), 1.0, 1.0)
        # )
        
        # --- Configuración del Emisor de Partículas ---
        self.particle_group = pygame.sprite.Group()
        self.particle_emitter = ParticleEmitter(self.particle_group, self.get_camera_offset)

        self.game.audio_manager.play_music()

    def get_camera_offset(self):
        return self.camara.camera_rect.topleft

    def update(self):
        if not self.game.running: return
        
        # Calculamos el movimiento de la cámara en este frame
        camera_dx = self.camara.camera_rect.x - self.last_camera_x
        self.last_camera_x = self.camara.camera_rect.x
        
        # Actualizamos todos los sistemas
        self.game.background_manager.update(camera_dx, 0) # 0 para scroll vertical
        self.sprite_groups["all_sprites"].update(plataformas=self.sprite_groups["collision_sprites"], audio_manager=self.game.audio_manager)
        self.particle_emitter.update()
        self.camara.update(self.jugador)

        # Colisiones
        enemigos_tocados = pygame.sprite.spritecollide(self.jugador, self.sprite_groups['enemies'], False)
        if enemigos_tocados:
            if self.jugador.vel.y > 0 and self.jugador.rect.bottom < enemigos_tocados[0].rect.centery + 10:
                enemy = enemigos_tocados[0]
                self.particle_emitter.explode(enemy.rect.centerx, enemy.rect.centery) # <-- ¡Explosión!
                enemy.kill()
                self.jugador.vel.y = -10
                self.game.audio_manager.play_sound('stomp')
            else:
                if self.jugador.hit(): self.game.audio_manager.play_sound('hit')
        
        # ... (resto de colisiones y lógica de vidas) ...

    def draw(self, surface):
        if not self.game.running: return
        
        # --- Orden de Dibujado (de atrás hacia adelante) ---
        # 1. Fondo (Cielo, montañas, etc.)
        self.game.background_manager.draw(surface, self.camara.camera_rect.topleft)
        
        # 2. Sprites del juego (Plataformas, Jugador, Enemigos)
        for sprite in sorted(self.sprite_groups["all_sprites"], key=lambda s: s.rect.bottom):
            surface.blit(sprite.image, self.camara.apply(sprite))
            
        # 3. Partículas (encima de todo)
        self.particle_emitter.draw(surface)
        
        # 4. UI (estática, no se mueve con la cámara)
        font = pygame.font.Font(None, 36)
        lives_text = font.render(f"Vidas: {self.jugador.vidas}", True, (255,255,255))
        surface.blit(lives_text, (10, 10))
            
    def handle_events(self, events):
        pass
```

Pygame no puede reproducir archivos `.mp4` o `.gif` directamente como un fondo. Lo que hacemos es convertir ese video en una secuencia de imágenes (ej: `frame_001.png`, `frame_002.png`, etc.) y luego le decimos a nuestro motor que las reproduzca una tras otra, creando la ilusión de un video.

-----

### **Paso 1: Convertir tu Video en Fotogramas (Frames)**

Antes de tocar el código, necesitamos la secuencia de imágenes. La herramienta estándar para esto es **FFmpeg**.

1.  **Descarga FFmpeg:** Si aún no lo tienes, descárgalo desde [ffmpeg.org](https://ffmpeg.org/download.html).

2.  **Obtén un Video:** Consigue un video corto (ej: `mi_video.mp4`).

3.  **Crea la Carpeta:** En tu proyecto, crea la carpeta: `video_demo/assets/video_frames/`.

4.  **Ejecuta el Comando:** Abre una terminal (CMD o PowerShell) en la carpeta que contiene tu `mi_video.mp4` y ejecuta este comando:

    ```bash
    ffmpeg -i mi_video.mp4 -vf "fps=30" C:\ruta\a\tu\proyecto\video_demo\assets\video_frames\frame_%04d.png
    ```

      * `-i mi_video.mp4`: El archivo de video de entrada.
      * `-vf "fps=30"`: Extrae 30 fotogramas por segundo.
      * `frame_%04d.png`: El patrón de salida. Esto creará `frame_0001.png`, `frame_0002.png`, etc., en tu carpeta `video_frames`.

-----

### **2. Los Módulos del Motor Necesarios**

Asegúrate de tener estos archivos en tu carpeta `engine/`.

#### **`engine/asset_manager.py` (Actualizado)**

*Añadimos el método `load_image_sequence`.*

```python
import pygame
import os
from .spritesheet import Spritesheet

class AssetManager:
    def __init__(self):
        self.spritesheets = {}
        self.images = {}
        self.sounds = {}

    def load_spritesheet(self, name, json_path):
        self.spritesheets[name] = Spritesheet(json_path)

    def load_image(self, name, image_path):
        self.images[name] = pygame.image.load(image_path).convert_alpha()
    
    def load_image_sequence(self, name, folder_path):
        """
        Carga una secuencia de imágenes (frames) desde una carpeta,
        ordenadas alfabéticamente.
        """
        frames = []
        try:
            filenames = sorted(os.listdir(folder_path))
            for filename in filenames:
                if filename.endswith(('.png', '.jpg', '.webp')):
                    image_path = os.path.join(folder_path, filename)
                    frames.append(pygame.image.load(image_path).convert()) # .convert() es más rápido para fondos
            
            if not frames:
                print(f"Advertencia: No se encontraron imágenes en la carpeta: {folder_path}")
            
            self.images[name] = frames # Guardamos la lista de frames
            print(f"Secuencia '{name}' cargada ({len(frames)} frames) desde '{folder_path}'")
        except Exception as e:
            print(f"Error al cargar la secuencia '{name}': {e}")
            raise

    def get_sprite(self, sheet_name, sprite_name):
        return self.spritesheets[sheet_name].get_sprite(sprite_name)

    def get_animation(self, sheet_name, anim_name):
        return self.spritesheets[sheet_name].get_animation_frames(anim_name)

    def get_image(self, name):
        return self.images[name]
```

#### **`engine/background.py` (¡NUEVO\!)**

*Contiene la lógica para el fondo animado.*

```python
import pygame

class ParallaxLayer:
    def __init__(self, image_surface, scroll_factor_x, scroll_factor_y):
        self.image = image_surface
        self.scroll_factor_x = scroll_factor_x
        self.scroll_factor_y = scroll_factor_y
        self.rect = self.image.get_rect()

    def update(self, camera_dx, camera_dy):
        self.rect.x -= camera_dx * self.scroll_factor_x
        self.rect.y -= camera_dy * self.scroll_factor_y

    def draw(self, surface, camera_offset):
        # Dibujamos la imagen dos veces para un bucle infinito (tiling)
        surface.blit(self.image, (self.rect.x + camera_offset.x, self.rect.y + camera_offset.y))
        surface.blit(self.image, (self.rect.x + self.rect.width + camera_offset.x, self.rect.y + camera_offset.y))
        
        # Resetear la posición para el bucle
        if self.rect.x + self.rect.width + camera_offset.x <= 0:
            self.rect.x = -camera_offset.x

class AnimatedBackgroundLayer(ParallaxLayer):
    """Una capa de fondo que es una animación (video pre-renderizado)."""
    def __init__(self, frames, scroll_factor_x, scroll_factor_y, anim_speed=0.5):
        self.frames = frames
        self.is_animated = True
        self.current_frame = 0
        self.anim_speed = anim_speed
        self.anim_timer = 0
        super().__init__(self.frames[0], scroll_factor_x, scroll_factor_y)

    def update(self, camera_dx, camera_dy):
        super().update(camera_dx, camera_dy)
        
        # Actualizar animación
        self.anim_timer += self.anim_speed
        if self.anim_timer >= 1:
            self.anim_timer = 0
            self.current_frame = (self.current_frame + 1) % len(self.frames)
            self.image = self.frames[self.current_frame]

class BackgroundManager:
    def __init__(self):
        self.layers = []

    def add_layer(self, layer):
        self.layers.append(layer)
        self.layers.sort(key=lambda l: (l.scroll_factor_x + l.scroll_factor_y) / 2)

    def update(self, camera_dx, camera_dy):
        for layer in self.layers:
            layer.update(camera_dx, camera_dy)

    def draw(self, surface, camera_offset):
        for layer in self.layers:
            layer.draw(surface, camera_offset)
```

#### **`engine/ui.py` (Necesario para la `Scene`)**

```python
# engine/ui.py
import pygame

class Scene:
    def __init__(self, game): self.game = game
    def handle_events(self, events): pass
    def update(self): pass
    def draw(self, surface): pass

class Button:
    # ... (código del botón) ...
```

-----

### **3. El Ejemplo de Juego (`video_demo/`)**

Crea una nueva carpeta `video_demo/` al mismo nivel que `engine/`.

#### **`video_demo/scenes/game_scene.py`**

*Esta escena carga y muestra el video.*

```python
# video_demo/scenes/game_scene.py
import pygame
from engine.ui import Scene
from engine.background import AnimatedBackgroundLayer

class GameScene(Scene):
    def __init__(self, game):
        super().__init__(game)
        
        # Obtenemos los fotogramas del video cargados por el AssetManager
        video_frames = self.game.assets.get_image('video_fondo')
        
        if video_frames:
            # Creamos la capa de fondo animada
            # Un factor de scroll de 0.0 significa que está fijo (no se mueve)
            video_layer = AnimatedBackgroundLayer(video_frames, 0.0, 0.0, anim_speed=0.5)
            self.game.background_manager.add_layer(video_layer)
        else:
            print("Error: No se encontraron fotogramas para la animación de fondo.")

    def update(self):
        # Actualizamos el gestor de fondos (que actualiza la animación)
        # 0, 0 porque la cámara no se mueve
        self.game.background_manager.update(0, 0)

    def draw(self, surface):
        surface.fill((0, 0, 0)) # Fondo negro por si acaso
        # Dibujamos el gestor de fondos
        self.game.background_manager.draw(surface, pygame.math.Vector2(0, 0))
            
    def handle_events(self, events):
        pass
```

#### **`video_demo/game.py`**

*El orquestador que carga los assets.*

```python
# video_demo/game.py
import pygame
import os
from engine.asset_manager import AssetManager
from engine.background import BackgroundManager
from .scenes.game_scene import GameScene

class VideoDemoGame:
    def __init__(self):
        pygame.init()
        self.screen = pygame.display.set_mode((1280, 720))
        pygame.display.set_caption("Demo de Fondo de Video")
        self.clock = pygame.time.Clock()
        self.running = True
        
        self.game_dir = os.path.dirname(__file__)
        self.assets_dir = os.path.join(self.game_dir, 'assets')
        
        self.assets = AssetManager()
        self.background_manager = BackgroundManager() # Creamos el gestor
        self.load_assets()
        
        if self.running:
            self.scenes = {'game': GameScene(self)}
            self.current_scene = self.scenes['game']

    def load_assets(self):
        try:
            video_frames_path = os.path.join(self.assets_dir, 'video_frames')
            self.assets.load_image_sequence('video_fondo', video_frames_path)
        except Exception as e:
            print(f"Error fatal al cargar assets: {e}")
            self.running = False
            
    def run(self):
        if not self.running: return
        while self.running:
            events = pygame.event.get()
            for event in events:
                if event.type == pygame.QUIT: self.running = False
            
            self.current_scene.handle_events(events)
            self.current_scene.update()
            self.current_scene.draw(self.screen)
            pygame.display.flip()
            self.clock.tick(60) # 60 FPS
        pygame.quit()

    def change_scene(self, scene_name):
        self.current_scene = self.scenes.get(scene_name)
```

#### **`video_demo/main.py` (El Punto de Entrada)**

```python
# video_demo/main.py
from .game import VideoDemoGame

def main():
    juego = VideoDemoGame()
    juego.run()

if __name__ == '__main__':
    main()
```

**Open Cv**

 El método de extraer fotogramas con FFmpeg funciona, pero es un proceso de "pre-renderizado".

Usar **OpenCV** (`cv2`) nos permite **decodificar el video en tiempo real**, fotograma a fotograma. Es una solución mucho más elegante y dinámica. La librería `Pillow` no es estrictamente necesaria para la conversión, ya que `OpenCV` (con `numpy`) y `Pygame` pueden comunicarse directamente, lo cual es más rápido.

Veremos cómo integrar la reproducción de video en tiempo real en nuestro motor.

-----

### **Clase: Fondos de Video en Tiempo Real con OpenCV**

**Objetivo:** Modificar nuestro motor para que pueda cargar, decodificar y reproducir archivos de video (`.mp4`, `.webp`, etc.) en tiempo real como un fondo de juego, utilizando la librería `opencv-python`.

-----

### **1. El Concepto: De Fotogramas a "Stream" de Video**

En lugar de cargar imágenes PNG en la memoria (lo cual es muy intensivo), abriremos un "stream" o flujo de video.

1.  **OpenCV (`cv2`):** Usaremos esta librería para abrir el archivo de video (`mi_video.mp4`).
2.  **Bucle del Juego:** En cada fotograma de nuestro juego, le pediremos a OpenCV: "dame el siguiente fotograma del video".
3.  **Conversión:** OpenCV nos da el fotograma en un formato de datos (`numpy array BGR`). Lo convertiremos al formato que Pygame entiende (`Surface RGB`).
4.  **Renderizado:** Pygame dibujará ese fotograma en la pantalla como si fuera cualquier otra imagen.

Esto es eficiente, ya que solo un fotograma del video existe en la memoria en un momento dado.

-----

### **2. Nuevas Dependencias (¡Importante\!)**

Este método requiere dos nuevas librerías. Abre tu terminal y ejecútalas:

```bash
pip install opencv-python
pip install numpy
```

**Tu `requirements.txt` ahora debería incluir:**

```
pygame
opencv-python
numpy
```

-----

### **3. Expansión del Motor**

Solo necesitamos actualizar un archivo en nuestro motor.

#### **Archivo: `engine/background.py` (Actualizado)**

Añadimos una nueva clase, `LiveVideoLayer`, que se encarga de esto.

```python
# engine/background.py
import pygame
import cv2  # Importamos OpenCV
import numpy # Importamos Numpy

class ParallaxLayer:
    """Clase base para una capa de fondo (sin cambios)."""
    def __init__(self, image_surface, scroll_factor_x, scroll_factor_y):
        self.image = image_surface
        self.scroll_factor_x = scroll_factor_x
        self.scroll_factor_y = scroll_factor_y
        self.rect = self.image.get_rect()

    def update(self, camera_dx, camera_dy):
        self.rect.x -= camera_dx * self.scroll_factor_x
        self.rect.y -= camera_dy * self.scroll_factor_y

    def draw(self, surface, camera_offset):
        surface.blit(self.image, (self.rect.x + camera_offset.x, self.rect.y + camera_offset.y))
        # (Aquí se podría añadir lógica de 'tiling' para fondos que se repiten)

class AnimatedBackgroundLayer(ParallaxLayer):
    """Una capa de fondo que es una animación de fotogramas pre-cargados (sin cambios)."""
    def __init__(self, frames, scroll_factor_x, scroll_factor_y, anim_speed=0.5):
        # ... (código de la clase anterior sin cambios) ...
        pass

# --- ¡NUEVA CLASE PARA VIDEO EN TIEMPO REAL! ---

class LiveVideoLayer(ParallaxLayer):
    """
    Una capa de fondo que decodifica un archivo de video en tiempo real usando OpenCV.
    """
    def __init__(self, video_path, scroll_factor_x, scroll_factor_y):
        self.cap = cv2.VideoCapture(video_path)
        if not self.cap.isOpened():
            raise FileNotFoundError(f"No se pudo abrir el archivo de video: {video_path}")
        
        # Obtenemos las propiedades del video
        self.width = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        
        # Leemos el primer fotograma para inicializar la clase padre
        ret, first_frame = self.cap.read()
        if not ret:
            raise ValueError(f"No se pudo leer el primer fotograma del video: {video_path}")
        
        # Convertimos ese primer fotograma a una Surface de Pygame
        initial_surface = self._convert_cv_to_pygame(first_frame)
        
        # Inicializamos la clase ParallaxLayer
        super().__init__(initial_surface, scroll_factor_x, scroll_factor_y)

    def _convert_cv_to_pygame(self, cv_frame):
        """Convierte un fotograma de OpenCV (BGR) a una Surface de Pygame (RGB)."""
        try:
            # OpenCV usa BGR, Pygame usa RGB. Los convertimos.
            cv_frame_rgb = cv2.cvtColor(cv_frame, cv2.COLOR_BGR2RGB)
            # Volteamos la imagen en el eje X (opcional, a veces es necesario)
            # cv_frame_rgb = cv2.flip(cv_frame_rgb, 0)
            
            # Creamos la superficie de Pygame directamente desde el buffer de numpy
            return pygame.image.frombuffer(cv_frame_rgb.tobytes(), (self.width, self.height), 'RGB')
        except Exception as e:
            print(f"Error al convertir fotograma: {e}")
            return self.image # Devuelve la última imagen buena

    def update(self, camera_dx, camera_dy):
        # Actualizamos la posición de scroll (si la hay)
        super().update(camera_dx, camera_dy)
        
        # Leemos el siguiente fotograma del video
        ret, frame = self.cap.read()
        
        if not ret:
            # Si el video terminó, lo rebobinamos al principio
            self.cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            ret, frame = self.cap.read()
        
        if ret:
            # Actualizamos la imagen de esta capa con el nuevo fotograma
            self.image = self._convert_cv_to_pygame(frame)

    def close(self):
        """Libera el recurso del video cuando ya no se necesita."""
        self.cap.release()

class BackgroundManager:
    """Gestiona todas las capas de fondo (sin cambios)."""
    def __init__(self):
        self.layers = []

    def add_layer(self, layer):
        self.layers.append(layer)
        self.layers.sort(key=lambda l: (l.scroll_factor_x + l.scroll_factor_y) / 2)

    def update(self, camera_dx, camera_dy):
        for layer in self.layers:
            layer.update(camera_dx, camera_dy)

    def draw(self, surface, camera_offset):
        for layer in self.layers:
            layer.draw(surface, camera_offset)
            
    def close_all(self):
        """Llama a 'close' en cualquier capa que lo necesite (como el video)."""
        for layer in self.layers:
            if hasattr(layer, 'close'):
                layer.close()

```

-----

### **4. Ejemplo de Juego: `video_demo_opencv`**

Ahora creamos un nuevo proyecto para usar esta funcionalidad.

**Estructura de Carpetas:**

```
/video_demo_opencv/
|-- assets/
|   |-- mi_video.mp4  <-- ¡DEBES COLOCAR TU VIDEO AQUÍ!
|
|-- scenes/
|   |-- game_scene.py
|
|-- game.py
|-- main.py
```

#### **Archivo: `video_demo_opencv/scenes/game_scene.py`**

```python
import pygame
from engine.ui import Scene
from engine.background import LiveVideoLayer
import os

class GameScene(Scene):
    def __init__(self, game):
        super().__init__(game)
        
        # Ruta al video
        video_path = os.path.join(self.game.assets_dir, 'mi_video.mp4')
        
        try:
            # Creamos la capa de video en tiempo real
            # Factor de scroll 0.0 para que esté fijo.
            self.video_layer = LiveVideoLayer(video_path, 0.0, 0.0)
            
            # Redimensionamos la ventana del juego al tamaño del video
            pygame.display.set_mode((self.video_layer.width, self.video_layer.height))
            
            # Añadimos la capa al gestor de fondos
            self.game.background_manager.add_layer(self.video_layer)
        except Exception as e:
            print(f"Error fatal al cargar el video: {e}")
            self.game.running = False

    def update(self):
        if not self.game.running: return
        self.game.background_manager.update(0, 0) # 0, 0 = sin scroll

    def draw(self, surface):
        if not self.game.running: return
        surface.fill((0, 0, 0))
        self.game.background_manager.draw(surface, pygame.math.Vector2(0, 0))
            
    def handle_events(self, events):
        pass
```

#### **Archivo: `video_demo_opencv/game.py`**

```python
import pygame
import os
from engine.asset_manager import AssetManager
from engine.background import BackgroundManager
from .scenes.game_scene import GameScene

class VideoDemoGame:
    def __init__(self):
        pygame.init()
        # Iniciamos con un tamaño temporal, la escena lo ajustará
        self.screen = pygame.display.set_mode((800, 600))
        pygame.display.set_caption("Demo de Fondo de Video con OpenCV")
        self.clock = pygame.time.Clock()
        self.running = True
        
        self.game_dir = os.path.dirname(__file__)
        self.assets_dir = os.path.join(self.game_dir, 'assets')
        
        self.assets = AssetManager() # No cargamos nada, pero la escena lo espera
        self.background_manager = BackgroundManager()
        
        if self.running:
            self.scenes = {'game': GameScene(self)}
            self.current_scene = self.scenes['game']

    def load_assets(self):
        # No necesitamos cargar assets de video aquí
        pass
            
    def run(self):
        if not self.running: return
        while self.running:
            events = pygame.event.get()
            for event in events:
                if event.type == pygame.QUIT:
                    self.running = False
            
            self.current_scene.handle_events(events)
            self.current_scene.update()
            self.current_scene.draw(self.screen)
            pygame.display.flip()
            self.clock.tick(60)
        
        # Limpieza importante al cerrar
        self.background_manager.close_all()
        pygame.quit()

    def change_scene(self, scene_name):
        self.current_scene = self.scenes.get(scene_name)
```

#### **Archivo: `video_demo_opencv/main.py`**

```python
# video_demo_opencv/main.py
from .game import VideoDemoGame

def main():
    juego = VideoDemoGame()
    juego.run()

if __name__ == '__main__':
    main()
```